# PII candidate filtering with Mistral

Ноутбук берёт `.jsonl` из `real_pii_collection/data/data_for_pii_finding/`, прогоняет сообщения через Mistral батчами и сразу сохраняет найденные PII-кандидаты.

Состояние обработки хранится на диск после каждого успешного батча, поэтому при падении API можно просто перезапустить ячейку обработки: она продолжит с последней сохранённой позиции.


In [6]:
ENABLE_PHOENIX = False

if ENABLE_PHOENIX:
    import phoenix as px
    from phoenix.otel import register
    from openinference.instrumentation.langchain import LangChainInstrumentor

    session = px.launch_app()
    tracer_provider = register()
    LangChainInstrumentor().instrument(tracer_provider=tracer_provider)
    print(f"Phoenix готов! Дашборд тут: {session.url}")
else:
    print("Phoenix disabled")


Phoenix disabled


In [7]:
from pathlib import Path
import json
import os
import re
import time

import pandas as pd
from dotenv import load_dotenv
from langchain_mistralai import ChatMistralAI
from langchain_core.messages import HumanMessage, SystemMessage
from pydantic import AliasChoices, BaseModel, Field
from typing import Literal

PROJECT_DIR = Path.cwd()
while PROJECT_DIR.name and not (PROJECT_DIR / "pyproject.toml").exists():
    PROJECT_DIR = PROJECT_DIR.parent

load_dotenv(PROJECT_DIR / ".env")
DATA_DIR = PROJECT_DIR / "real_pii_collection" / "data"
PII_INPUT_DIR = DATA_DIR / "data_for_pii_finding"
STATE_DIR = DATA_DIR / "llm_state"
OUTPUT_DIR = DATA_DIR / "outputs"
CANDIDATES_DIR = OUTPUT_DIR / "llm_candidates"
for directory in [PII_INPUT_DIR, STATE_DIR, CANDIDATES_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

# Если None, берётся первый *.jsonl из PII_INPUT_DIR по алфавиту. Можно задать имя файла явно, например "dvach_posts.jsonl".
PII_INPUT_FILE_NAME = None

MODEL_NAME = os.getenv("MISTRAL_MODEL", "mistral-medium-latest")
print("MISTRAL_API_KEY loaded:", bool(os.getenv("MISTRAL_API_KEY")))
print("model:", MODEL_NAME)
print("pii_input_dir:", PII_INPUT_DIR)


MISTRAL_API_KEY loaded: True
model: mistral-medium-latest
pii_input_dir: /Users/artemzmailov/Desktop/kitoboy-PII/real_pii_collection/data/data_for_pii_finding


In [8]:
queue_files = sorted(PII_INPUT_DIR.glob("*.jsonl"))
if PII_INPUT_FILE_NAME is not None:
    input_jsonl = PII_INPUT_DIR / PII_INPUT_FILE_NAME
    if not input_jsonl.exists():
        raise FileNotFoundError(f"Queue file not found: {input_jsonl}")
else:
    if not queue_files:
        raise FileNotFoundError(f"Положи *.jsonl файл в {PII_INPUT_DIR}")
    input_jsonl = queue_files[0]

state_path = STATE_DIR / f"{input_jsonl.stem}.state.json"
candidates_jsonl_path = CANDIDATES_DIR / f"{input_jsonl.stem}.candidates.jsonl"
candidates_csv_path = CANDIDATES_DIR / f"{input_jsonl.stem}.candidates.csv"

def count_lines(path: Path) -> int:
    with path.open("rb") as f:
        return sum(1 for _ in f)

total_file_lines = count_lines(input_jsonl)

print("input:", input_jsonl)
print("state:", state_path)
print("candidates_jsonl:", candidates_jsonl_path)
print("total_file_lines:", total_file_lines)
print("queue files:", [p.name for p in queue_files])


input: /Users/artemzmailov/Desktop/kitoboy-PII/real_pii_collection/data/data_for_pii_finding/dvach_posts_keywords.jsonl
state: /Users/artemzmailov/Desktop/kitoboy-PII/real_pii_collection/data/llm_state/dvach_posts_keywords.state.json
candidates_jsonl: /Users/artemzmailov/Desktop/kitoboy-PII/real_pii_collection/data/outputs/llm_candidates/dvach_posts_keywords.candidates.jsonl
total_file_lines: 6569
queue files: ['dvach_posts_keywords.jsonl']


In [9]:
MAX_ROWS = 0  # 0 = обработать файл до конца; удобно поставить 1000 для теста
BATCH_SIZE = 50
MIN_TEXT_LEN = 20
MAX_TEXT_LEN = 3500

def load_state(path: Path) -> dict:
    if not path.exists():
        return {"next_line_idx": 0, "processed_rows": 0, "positive_rows": 0}
    return json.loads(path.read_text(encoding="utf-8"))

def save_state(path: Path, state: dict) -> None:
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(state, ensure_ascii=False, indent=2), encoding="utf-8")
    tmp.replace(path)

def iter_jsonl_from(path: Path, start_line_idx: int):
    with path.open("r", encoding="utf-8") as f:
        for line_idx, line in enumerate(f):
            if line_idx < start_line_idx or not line.strip():
                continue
            row = json.loads(line)
            text = str(row.get("text") or "")
            if not (MIN_TEXT_LEN <= len(text) <= MAX_TEXT_LEN):
                yield line_idx, row, "skip_length"
            else:
                yield line_idx, row, None

def append_jsonl(path: Path, records: list[dict]) -> None:
    if not records:
        return
    with path.open("a", encoding="utf-8") as f:
        for record in records:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")

state = load_state(state_path)
next_line_idx = int(state.get("next_line_idx", 0))
remaining_lines = max(total_file_lines - next_line_idx, 0)
max_rows_this_run = remaining_lines if MAX_ROWS == 0 else min(MAX_ROWS, remaining_lines)
estimated_batches = (max_rows_this_run + BATCH_SIZE - 1) // BATCH_SIZE
print("loaded_state:", state)
print(f"progress: next_line_idx={next_line_idx}/{total_file_lines} ({next_line_idx / max(total_file_lines, 1):.1%})")
print(f"remaining_lines={remaining_lines} | max_rows_this_run={max_rows_this_run} | estimated_batches={estimated_batches} | batch_size={BATCH_SIZE}")


loaded_state: {'next_line_idx': 0, 'processed_rows': 0, 'positive_rows': 0}
progress: next_line_idx=0/6569 (0.0%)
remaining_lines=6569 | max_rows_this_run=6569 | estimated_batches=132 | batch_size=50


In [12]:
ENTITY_NAMES = [
    # Личные документы
    "Паспорт РФ",
    "Загранпаспорт",
    "Водительское удостоверение",
    "СНИЛС",
    "ИНН физического лица",
    "ИНН юридического лица / ИП",
    "Военный билет",
    "Полис ОМС",
    "Полис ДМС",
    "Удостоверение беженца",
    "Вид на жительство",
    "Удостоверение ветерана боевых действий",
    "Удостоверение ветерана труда",
    "Удостоверение многодетной семьи",
    "Удостоверение личности моряка",
    "Трудовая книжка / электронная трудовая",

    # Адресно-географические данные
    "Адрес регистрации, проживания или фактического нахождения",
    "Город или населенный пункт",
    "Регион, область, край, страна проживания или нахождения",

    # Персонально-биографические данные
    "ФИО, имя и фамилия, имя с отчеством, инициалы с фамилией",
    "Возраст",
    "Дата рождения",
    "Пол",
    "Семейное положение",
    "Наличие детей",
    "Образование",
    "Место работы",
    "Должность",
    "Профессия",
    "Рост и вес",

    # Медицинские данные
    "Инвалидность и степень инвалидности",
    "Диагнозы и медицинские состояния",
    "Аллергии",

    # Финансовые и банковские идентификаторы
    "Справка 2-НДФЛ или справка о доходах",
    "Банковская карта",
    "Банковский счет",
    "ЮMoney / Яндекс деньги",
    "Криптокошелек или адрес криптокошелька",
    "SWIFT/BIC код",
    "IBAN",
    "WebMoney",

    # Транспортные идентификаторы
    "VIN автомобиля",
    "Госномер автомобиля",
    "Номер ПТС",
    "Номер СТС",
    "Номер полиса ОСАГО",
    "Номер полиса КАСКО",

    # Документы об образовании
    "Диплом с номером или серией",
    "Аттестат",
    "Студенческий билет",
    "Ученический билет",
    "Ученое звание или ученая степень",

    # Судебные и исполнительные идентификаторы
    "Номер исполнительного производства",
    "Номер исполнительного документа",
    "Номер арбитражного дела",
    "Номер гражданского, уголовного или административного дела",
    "Номер судебного приказа",
    "Номер дела в суде общей юрисдикции",

    # Социальные сети и мессенджеры
    "Telegram",
    "VK / ВКонтакте / vk.me",
    "WhatsApp",
    "Discord",
    "Twitter / X",
    "Instagram",
    "TikTok",
    "YouTube канал или профиль",
    "Skype",
    "Signal",
    "Viber",
    "Max",
    "Github",

    # Игровые платформы
    "Steam профиль или Steam ID",
    "Epic Games профиль или ник",
    "Battle.net профиль, BattleTag или ник",
    "PlayStation Network профиль или PSN Online ID",
    "Xbox Live профиль или ник",
    "Roblox профиль, User ID или username",
    "Genshin Impact UID или ник",

    # Карьерные платформы
    "hh.ru / HeadHunter профиль, резюме или аккаунт",
    "Habr Карьера профиль",
    "LinkedIn профиль",
    "SuperJob профиль или резюме",
    "Авито Работа профиль, резюме или объявление конкретного человека",

    # Семейные и социальные свидетельства
    "Свидетельство о заключении или расторжении брака",
    "Свидетельство о рождении ребенка",
    "Справка об отсутствии или наличии судимости",

    # Трек-номера
    "Трек-номер Почты России",
    "Международный трек-номер",
    "Трек-номер EMS",
    "Международный S10 трек-номер",

    # Цифровые идентификаторы
    "Телефон",
    "Email",
    "Никнейм, username или ID аккаунта",
    "IP-адрес",
]

ENTITY_LIST_TEXT = "\n".join(f"- {name}" for name in ENTITY_NAMES)
print(f"entity names in prompt: {len(ENTITY_NAMES)}")
print(ENTITY_LIST_TEXT)


class PiiCandidate(BaseModel):
    id: str = Field(
        validation_alias=AliasChoices("id", "id.id"),
        description="ID входного сообщения. Принимаем также id.id, потому что Mistral иногда криво называет ключ."
    )
    has_pii: bool = Field(default=True, description="True для возвращённых кандидатов")
    pii_types: list[str] = Field(default_factory=list, description="Типы найденных персональных данных из списка или похожего типа")
    comment: str = Field(default="", description="Коротко: какие конкретные значения сущностей есть в тексте")


class PiiBatchResult(BaseModel):
    candidates: list[PiiCandidate] = Field(default_factory=list, description="Только сообщения, где есть конкретные значения персональных сущностей")


SYSTEM_PROMPT_TEMPLATE = """
Ты специалист по поиску персональных данных в пользовательских текстах.

Мы собираем кандидаты для ручной разметки PII. Твоя задача — вернуть только те сообщения,
где прямо написано конкретное значение персональной сущности.

Ниже список типов сущностей, которые нас интересуют. Список не исчерпывающий: если в тексте есть похожий
персональный идентификатор или персональный факт такого же рода, его тоже можно отобрать.

{entity_list}

Что считать кандидатом:
- в тексте есть конкретное значение сущности;
- значение дословно присутствует в тексте;
- по нему можно было бы разметить span в исходном сообщении;
- это может быть как жесткий идентификатор (телефон, email, Telegram, документ, реквизит, профиль),
  так и конкретный персональный факт из списка (возраст, город, профессия, место работы, образование, диагноз и т.п.).

Примеры кандидатов:
- "мой тг @username" — Telegram;
- "+7..." — телефон;
- "Иванов Иван Иванович" — ФИО;
- "мне 35" — возраст;
- "Дима, Москва, 36 лет" — имя, город, возраст;
- "работаю токарем" — профессия;
- "живу в Ижевске" — город;
- "СНИЛС ..." — СНИЛС;
- "ИНН ..." — ИНН;
- "адрес: ..." — адрес;
- "hh.ru/resume/..." — карьерный профиль/резюме.

Что НЕ считать кандидатом:
- обсуждение типа сущности без конкретного значения: "паспорт нужен", "СНИЛС оформляют", "телефон сел";
- общее рассуждение о профессиях, документах, болезнях, городах, зарплатах без конкретного значения сущности;
- публичных людей, бренды, персонажей, названия произведений;
- обычные вакансии, новости, видео или статьи, если это не личный контакт, профиль, резюме или данные конкретного человека;
- догадки и выводы, которых нет дословно в тексте.

Важно:
- возвращай только id сообщений-кандидатов;
- в pii_types укажи типы найденных сущностей;
- в comment коротко объясни, какие конкретные значения есть в тексте;
- не возвращай сообщения без конкретных значений сущностей;
- если в батче нет кандидатов, верни пустой список candidates.
""".strip()

SYSTEM_PROMPT = SYSTEM_PROMPT_TEMPLATE.format(entity_list=ENTITY_LIST_TEXT)


llm = ChatMistralAI(model=MODEL_NAME, temperature=0.0, timeout=120)
structured_llm = llm.with_structured_output(PiiBatchResult)


def classify_batch(batch_rows: list[dict]) -> list[dict]:
    payload = [{"id": str(item["id"]), "text": str(item["text"])} for item in batch_rows]
    messages = [
        SystemMessage(content=SYSTEM_PROMPT),
        HumanMessage(content=json.dumps(payload, ensure_ascii=False)),
    ]
    parsed = structured_llm.invoke(messages)
    return [candidate.model_dump() for candidate in parsed.candidates]


entity names in prompt: 94
- Паспорт РФ
- Загранпаспорт
- Водительское удостоверение
- СНИЛС
- ИНН физического лица
- ИНН юридического лица / ИП
- Военный билет
- Полис ОМС
- Полис ДМС
- Удостоверение беженца
- Вид на жительство
- Удостоверение ветерана боевых действий
- Удостоверение ветерана труда
- Удостоверение многодетной семьи
- Удостоверение личности моряка
- Трудовая книжка / электронная трудовая
- Адрес регистрации, проживания или фактического нахождения
- Город или населенный пункт
- Регион, область, край, страна проживания или нахождения
- ФИО, имя и фамилия, имя с отчеством, инициалы с фамилией
- Возраст
- Дата рождения
- Пол
- Семейное положение
- Наличие детей
- Образование
- Место работы
- Должность
- Профессия
- Рост и вес
- Инвалидность и степень инвалидности
- Диагнозы и медицинские состояния
- Аллергии
- Справка 2-НДФЛ или справка о доходах
- Банковская карта
- Банковский счет
- ЮMoney / Яндекс деньги
- Криптокошелек или адрес криптокошелька
- SWIFT/BIC код
- IBAN
- 

In [13]:
state = load_state(state_path)
start_line_idx = int(state.get("next_line_idx", 0))
processed_this_run = 0
positive_this_run = 0
batch = []
batch_line_idxs = []
batch_no = 0

def process_batch(batch: list[dict], batch_line_idxs: list[int]) -> None:
    global batch_no, processed_this_run, positive_this_run, state
    if not batch:
        return

    batch_no += 1
    first_line = batch_line_idxs[0]
    last_line = batch_line_idxs[-1]
    print(
        f"starting batch {batch_no} | file_lines {first_line + 1}-{last_line + 1} | "
        f"processed_this_run={processed_this_run}",
        flush=True,
    )

    t0 = time.perf_counter()
    try:
        verdicts = classify_batch(batch)  # intentionally fail-fast: do not skip on API/JSON errors
    except Exception as exc:
        debug_dir = STATE_DIR / "debug"
        debug_dir.mkdir(parents=True, exist_ok=True)
        debug_path = debug_dir / f"{input_jsonl.stem}.failed_batch_{first_line + 1}_{last_line + 1}.json"
        debug_payload = {
            "error": repr(exc),
            "first_line": first_line,
            "last_line": last_line,
            "batch": batch,
            "saved_at": time.strftime("%Y-%m-%d %H:%M:%S"),
        }
        debug_path.write_text(json.dumps(debug_payload, ensure_ascii=False, indent=2), encoding="utf-8")
        print(f"saved failed batch debug: {debug_path}", flush=True)
        raise
    batch_ids = {str(item["id"]) for item in batch}
    verdict_by_id = {
        str(item.get("id")): item
        for item in verdicts
        if str(item.get("id")) in batch_ids and item.get("has_pii") is True
    }

    positives = []
    for item in batch:
        verdict = verdict_by_id.get(str(item["id"]))
        if verdict is None:
            continue
        out = dict(item)
        out.update({
            "llm_model": MODEL_NAME,
            "llm_has_pii": True,
            "llm_pii_types": verdict.get("pii_types", []),
            "llm_comment": verdict.get("comment", ""),
            "source_pii_input_file": input_jsonl.name,
        })
        positives.append(out)

    append_jsonl(candidates_jsonl_path, positives)

    processed_this_run += len(batch)
    positive_this_run += len(positives)
    state.update({
        "pii_input_file": input_jsonl.name,
        "model": MODEL_NAME,
        "next_line_idx": last_line + 1,
        "processed_rows": int(state.get("processed_rows", 0)) + len(batch),
        "positive_rows": int(state.get("positive_rows", 0)) + len(positives),
        "last_batch_finished_at": time.strftime("%Y-%m-%d %H:%M:%S"),
        "candidates_jsonl": str(candidates_jsonl_path),
    })
    save_state(state_path, state)

    dt = time.perf_counter() - t0
    print(
        f"done batch {batch_no} | next_line_idx={state['next_line_idx']} | "
        f"positives={len(positives)} | total_positives={state['positive_rows']} | "
        f"{dt:.1f}s | {len(batch)/max(dt, 1e-9):.2f} posts/s",
        flush=True,
    )

for line_idx, row, skip_reason in iter_jsonl_from(input_jsonl, start_line_idx):
    if skip_reason:
        state.update({
            "pii_input_file": input_jsonl.name,
            "next_line_idx": line_idx + 1,
            "skipped_rows": int(state.get("skipped_rows", 0)) + 1,
            "last_skip_reason": skip_reason,
        })
        save_state(state_path, state)
        continue

    batch.append(row)
    batch_line_idxs.append(line_idx)
    if len(batch) >= BATCH_SIZE:
        process_batch(batch, batch_line_idxs)
        batch = []
        batch_line_idxs = []
        if MAX_ROWS and processed_this_run >= MAX_ROWS:
            print(f"stopped by MAX_ROWS={MAX_ROWS}", flush=True)
            break

if batch and (not MAX_ROWS or processed_this_run < MAX_ROWS):
    process_batch(batch, batch_line_idxs)

print(
    f"run finished | processed_this_run={processed_this_run} | positive_this_run={positive_this_run} | "
    f"next_line_idx={state.get('next_line_idx')} | candidates={candidates_jsonl_path}",
    flush=True,
)


starting batch 1 | file_lines 252-301 | processed_this_run=0
done batch 1 | next_line_idx=301 | positives=4 | total_positives=31 | 7.4s | 6.73 posts/s
starting batch 2 | file_lines 302-353 | processed_this_run=50
done batch 2 | next_line_idx=353 | positives=4 | total_positives=35 | 7.5s | 6.70 posts/s
starting batch 3 | file_lines 354-403 | processed_this_run=100
done batch 3 | next_line_idx=403 | positives=11 | total_positives=46 | 16.8s | 2.97 posts/s
starting batch 4 | file_lines 405-454 | processed_this_run=150
done batch 4 | next_line_idx=454 | positives=4 | total_positives=50 | 8.1s | 6.15 posts/s
starting batch 5 | file_lines 455-505 | processed_this_run=200
done batch 5 | next_line_idx=505 | positives=5 | total_positives=55 | 10.0s | 5.02 posts/s
starting batch 6 | file_lines 506-557 | processed_this_run=250
done batch 6 | next_line_idx=557 | positives=5 | total_positives=60 | 8.0s | 6.28 posts/s
starting batch 7 | file_lines 558-608 | processed_this_run=300
done batch 7 | next

In [ ]:
candidate_rows = []
if candidates_jsonl_path.exists():
    with candidates_jsonl_path.open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                candidate_rows.append(json.loads(line))

candidates = pd.DataFrame(candidate_rows)
if len(candidates):
    risk_order = {"high": 3, "medium": 2, "low": 1}
    candidates["risk_rank"] = candidates["llm_risk"].map(risk_order).fillna(0)
    candidates = candidates.sort_values(["risk_rank", "text_len"], ascending=[False, True]).drop(columns=["risk_rank"])
    flat = candidates.copy()
    flat["llm_pii_types"] = flat["llm_pii_types"].map(lambda x: ", ".join(x) if isinstance(x, list) else x)
    flat.to_csv(candidates_csv_path, index=False)

print("candidates:", len(candidates))
print("jsonl:", candidates_jsonl_path)
print("csv:", candidates_csv_path)


In [ ]:
pd.set_option("display.max_colwidth", 500)
if len(candidates):
    display(candidates[["board", "post_num", "llm_risk", "llm_pii_types", "text"]].head(20))
else:
    print("No candidates yet")
